# Proyecto final: identificación de raza mediante análisis de señales

Este proyecto usa el dataset DogSpeak y sigue directamente las técnicas de los laboratorios: muestreo, convolución, estadística, correlación y Transformada de Fourier.

No se utilizan Inteligencia Artificial, Machine Learning, Deep Learning, redes neuronales ni `sklearn`. La raza se determina comparando una señal de prueba con cinco patrones obtenidos mediante promedios de señales reales.

## Objetivo y flujo

El flujo completo es:

**Muestreo → Convolución → Estadística → Promedio por perro → Promedio por raza → Correlación → Fourier → Comparación de las cinco razas**

Cada vocalización se convierte a mono, se remuestrea a la misma frecuencia y longitud, y se normaliza. Luego se aplica un filtro de media móvil mediante convolución. Las señales filtradas se promedian por perro y esos promedios se vuelven a promediar por raza.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from scipy.signal import convolve, resample
from scipy.stats import norm
from IPython.display import Audio, display

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

metadata_locations = [
    Path('metadata.csv'),
    Path('Final/metadata.csv'),
    Path('DogSpeak/metadata.csv'),
    Path('/content/drive/MyDrive/DogSpeak/metadata.csv'),
    Path('/content/drive/MyDrive/metadata.csv'),
]
for path in metadata_locations:
    if path.exists():
        METADATA_PATH = path
        break
else:
    METADATA_PATH = Path('metadata.csv')

AUDIO_ROOT = Path('/content/drive/MyDrive/dogspeak_released')
if not AUDIO_ROOT.exists():
    AUDIO_ROOT = METADATA_PATH.parent / 'dogspeak_released'
if not AUDIO_ROOT.exists():
    AUDIO_ROOT = Path('dogspeak_released')

TARGET_SAMPLE_RATE = 8000
N_SAMPLES = TARGET_SAMPLE_RATE
MAX_FILES_PER_DOG = 20
FILTER_LENGTH = 31

print('Metadata:', METADATA_PATH.resolve())
print('Audios:', AUDIO_ROOT.resolve())
print('Metadata disponible:', METADATA_PATH.exists())

ModuleNotFoundError: No module named 'numpy'

In [ ]:
def load_metadata(metadata_path):
    metadata = pd.read_csv(metadata_path)
    required_columns = {'filename', 'breed', 'sex', 'dog_id'}
    if not required_columns.issubset(metadata.columns):
        raise ValueError(f'El CSV debe contener: {sorted(required_columns)}')
    metadata['breed'] = metadata['breed'].str.strip().str.lower()
    metadata['dog_id'] = metadata['dog_id'].astype(str)
    return metadata

def resolve_audio_path(audio_root, filename, dog_id):
    filename = Path(str(filename)).name
    candidates = [audio_root / dog_id / filename, audio_root / filename]
    for audio_path in candidates:
        if audio_path.exists():
            return audio_path
    matches = list(audio_root.rglob(filename))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'No se encontró el audio: {filename}')

metadata = load_metadata(METADATA_PATH)
print('Registros:', len(metadata))
print('Razas:', sorted(metadata['breed'].unique()))
display(metadata.head())

## 1. Lab 1: muestreo de la señal

Las grabaciones pueden tener distintas frecuencias de muestreo y duraciones. Para poder compararlas, se convierten a mono y se remuestrean a `8000 Hz` con exactamente `8000` muestras. Esta es la señal discreta que se usará en todo el proyecto.

In [ ]:
def sample_audio(audio_path):
    signal, _ = sf.read(audio_path, always_2d=False)
    signal = np.asarray(signal, dtype=float)
    if signal.ndim == 2:
        signal = signal.mean(axis=1)
    signal = signal - signal.mean()
    if np.max(np.abs(signal)) > 0:
        signal = signal / np.max(np.abs(signal))
    signal = resample(signal, N_SAMPLES)
    signal = signal - signal.mean()
    if np.max(np.abs(signal)) > 0:
        signal = signal / np.max(np.abs(signal))
    return signal

if metadata is not None:
    sample_row = metadata.iloc[0]
    sample_path = resolve_audio_path(AUDIO_ROOT, sample_row['filename'], sample_row['dog_id'])
    sampled_signal = sample_audio(sample_path)
    time_axis = np.arange(N_SAMPLES) / TARGET_SAMPLE_RATE
    figure, axes = plt.subplots(2, 1, figsize=(12, 6))
    axes[0].plot(time_axis, sampled_signal)
    axes[0].set(title=f'Señal muestreada: {sample_row["breed"]}', xlabel='Tiempo [s]', ylabel='Amplitud')
    axes[1].plot(time_axis[:1000], sampled_signal[:1000])
    axes[1].set(title='Detalle de las primeras 1000 muestras', xlabel='Tiempo [s]', ylabel='Amplitud')
    plt.tight_layout()
    plt.show()
    display(Audio(sampled_signal, rate=TARGET_SAMPLE_RATE))

## 2. Lab 2: reducción de ruido por convolución

Se usa un filtro de media móvil. Su respuesta impulsiva es una ventana rectangular normalizada y la señal filtrada se obtiene con `np.convolve(..., mode='same')`. Esto reduce variaciones rápidas y ruido de alta frecuencia sin usar un modelo aprendido.

In [ ]:
def moving_average_filter(signal, filter_length=FILTER_LENGTH):
    impulse_response = np.ones(filter_length) / filter_length
    filtered_signal = convolve(signal, impulse_response, mode='same')
    maximum = np.max(np.abs(filtered_signal), initial=0)
    if maximum > 0:
        filtered_signal = filtered_signal / maximum
    return filtered_signal, impulse_response

if metadata is not None:
    filtered_signal, impulse_response = moving_average_filter(sampled_signal)
    time_axis = np.arange(N_SAMPLES) / TARGET_SAMPLE_RATE
    figure, axes = plt.subplots(3, 1, figsize=(12, 8))
    axes[0].plot(time_axis, sampled_signal, color='gray')
    axes[0].set_title('Señal muestreada con ruido')
    axes[1].stem(impulse_response, basefmt=' ', linefmt='tab:orange', markerfmt='tab:orange')
    axes[1].set_title('Respuesta impulsiva del filtro de convolución')
    axes[2].plot(time_axis, filtered_signal, color='tab:green')
    axes[2].set_title('Señal después de la convolución')
    for axis in axes:
        axis.set_xlabel('Tiempo o muestra')
        axis.set_ylabel('Amplitud')
    plt.tight_layout()
    plt.show()

## 3. Lab 3: estadística, LGN y TLC

Se calculan media, varianza y desviación estándar de las señales. Además, se ilustran:

- **Ley de los Grandes Números:** el promedio de muchas variables Uniformes se acerca a su esperanza.
- **Teorema del Límite Central:** la suma de variables Uniformes se aproxima a una distribución Gaussiana cuando aumenta el número de sumandos.

Estas simulaciones explican por qué los promedios de muchas vocalizaciones pueden representar un comportamiento típico.

In [ ]:
def signal_statistics(signal):
    return {
        'media': np.mean(signal),
        'varianza': np.var(signal),
        'desviacion_estandar': np.std(signal),
        'energia_media': np.mean(signal ** 2)
    }

if metadata is not None:
    print('Estadísticos de la señal de ejemplo:')
    display(pd.DataFrame([signal_statistics(sampled_signal)]).round(6))

# Ley de los Grandes Números con variables Uniformes U(0, 1)
uniform_sample = rng.uniform(0, 1, 10000)
uniform_means = np.cumsum(uniform_sample) / np.arange(1, len(uniform_sample) + 1)

# Teorema del Límite Central: sumas de n Uniformes U(0, 1)
central_limit_sizes = [2, 5, 30, 100]
central_limit_sums = {}
for sum_size in central_limit_sizes:
    central_limit_sums[sum_size] = rng.uniform(0, 1, (5000, sum_size)).sum(axis=1)

figure, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(uniform_means, color='tab:blue')
axes[0, 0].axhline(0.5, color='red', linestyle='--', label='E[U] = 0.5')
axes[0, 0].set(title='Ley de los Grandes Números', xlabel='Cantidad de muestras', ylabel='Promedio')
axes[0, 0].legend()
for axis, sum_size in zip(axes.flat[1:], central_limit_sizes[:3]):
    values = central_limit_sums[sum_size]
    axis.hist(values, bins=35, density=True, alpha=0.7, color='tab:orange')
    theoretical_mean = sum_size / 2
    theoretical_std = np.sqrt(sum_size / 12)
    x_values = np.linspace(values.min(), values.max(), 200)
    axis.plot(x_values, norm.pdf(x_values, theoretical_mean, theoretical_std), 'r--')
    axis.set_title(f'TLC: suma de {sum_size} Uniformes')
    axis.set_xlabel('Suma')
    axis.set_ylabel('Densidad')
plt.tight_layout()
plt.show()

print('Uniforme U(0,1): media teórica = 0.5, varianza teórica = 1/12 =', round(1 / 12, 5))
for sum_size in central_limit_sizes:
    values = central_limit_sums[sum_size]
    print(f'n={sum_size}: media={values.mean():.4f}, varianza={values.var():.4f}, desviación={values.std():.4f}')

## 4. Promedio de las señales de cada perro

Para cada perro se seleccionan como máximo `MAX_FILES_PER_DOG` vocalizaciones. Cada audio se muestrea y filtra de la misma forma. Después se calcula:

\[
\bar{x}_{perro}(n) = \frac{1}{M}\sum_{k=1}^{M}x_k(n)
\]

Así, el patrón de un perro no depende de una sola grabación.

In [ ]:
def preprocess_audio(audio_path):
    sampled_signal = sample_audio(audio_path)
    filtered_signal, _ = moving_average_filter(sampled_signal)
    return filtered_signal

def build_dog_averages(metadata, audio_root):
    dog_signals = {}
    dog_breeds = {}
    dog_audio_counts = {}
    for dog_id, dog_rows in metadata.groupby('dog_id'):
        selected_rows = dog_rows.sample(min(MAX_FILES_PER_DOG, len(dog_rows)), random_state=RANDOM_SEED)
        accumulated_signal = np.zeros(N_SAMPLES)
        valid_count = 0
        for _, row in selected_rows.iterrows():
            try:
                audio_path = resolve_audio_path(audio_root, row['filename'], dog_id)
                accumulated_signal += preprocess_audio(audio_path)
                valid_count += 1
                dog_breeds[dog_id] = row['breed']
            except (OSError, ValueError, RuntimeError, FileNotFoundError) as error:
                print(f'Audio omitido: {row["filename"]} -> {error}')
        if valid_count:
            dog_signals[dog_id] = accumulated_signal / valid_count
            dog_audio_counts[dog_id] = valid_count
    return dog_signals, dog_breeds, dog_audio_counts

if metadata is not None:
    dog_signals, dog_breeds, dog_audio_counts = build_dog_averages(metadata, AUDIO_ROOT)
    print('Perros procesados:', len(dog_signals))
    display(pd.Series(dog_audio_counts).describe().to_frame('cantidad').round(2))

## 5. Promedio por raza

El patrón de cada raza se obtiene promediando los patrones de sus perros:

\[
\bar{x}_{raza}(n) = \frac{1}{D}\sum_{d=1}^{D}\bar{x}_{perro,d}(n)
\]

Este paso usa únicamente promedios de señales y conserva el significado físico del proyecto.

In [ ]:
def build_breed_averages(dog_signals, dog_breeds):
    breed_signals = {}
    for breed_name in sorted(set(dog_breeds.values())):
        breed_dogs = [dog_id for dog_id, breed in dog_breeds.items() if breed == breed_name]
        breed_signals[breed_name] = np.mean([dog_signals[dog_id] for dog_id in breed_dogs], axis=0)
    return breed_signals

if metadata is not None and 'dog_signals' in locals() and dog_signals:
    breed_signals = build_breed_averages(dog_signals, dog_breeds)
    print('Razas con patrón calculado:', sorted(breed_signals))
    time_axis = np.arange(N_SAMPLES) / TARGET_SAMPLE_RATE
    figure, axis = plt.subplots(figsize=(14, 5))
    for breed_name, breed_signal in breed_signals.items():
        axis.plot(time_axis, breed_signal, label=breed_name)
    axis.set(title='Patrones promedio en el dominio del tiempo', xlabel='Tiempo [s]', ylabel='Amplitud promedio')
    axis.legend()
    axis.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 6. Correlación, Fourier y decisión para una señal de prueba

La correlación cruzada se calcula entre la señal de prueba y cada patrón de raza. Como en el Lab 2, se permite un desplazamiento temporal y se toma la mayor correlación normalizada. La raza estimada es la que tenga el mayor valor.

También se calcula la FFT de la señal de prueba y de cada patrón. La gráfica de magnitudes permite comparar visualmente las bandas frecuenciales dominantes de las cinco razas.

In [ ]:
TEST_AUDIO_PATH = None

def normalized_correlation(test_signal, reference_signal):
    centered_test = test_signal - np.mean(test_signal)
    centered_reference = reference_signal - np.mean(reference_signal)
    correlation = np.correlate(centered_test, centered_reference, mode='full')
    denominator = np.linalg.norm(centered_test) * np.linalg.norm(centered_reference)
    normalized = correlation / (denominator + 1e-12)
    peak_index = np.argmax(np.abs(normalized))
    lag = peak_index - (len(reference_signal) - 1)
    return normalized[peak_index], lag, normalized

def positive_fft(signal):
    spectrum = np.abs(np.fft.rfft(signal))
    frequencies = np.fft.rfftfreq(len(signal), 1 / TARGET_SAMPLE_RATE)
    spectrum = spectrum / np.max(spectrum, initial=1e-12)
    return frequencies, spectrum

if metadata is not None and 'breed_signals' in locals() and breed_signals:
    if TEST_AUDIO_PATH is None:
        test_row = metadata.sample(1, random_state=RANDOM_SEED).iloc[0]
        TEST_AUDIO_PATH = resolve_audio_path(AUDIO_ROOT, test_row['filename'], test_row['dog_id'])
    else:
        TEST_AUDIO_PATH = Path(TEST_AUDIO_PATH)
    test_signal = preprocess_audio(TEST_AUDIO_PATH)
    correlation_results = []
    for breed_name, breed_signal in breed_signals.items():
        correlation_value, lag, _ = normalized_correlation(test_signal, breed_signal)
        correlation_results.append({'raza': breed_name, 'correlacion': correlation_value, 'retardo_muestras': lag})
    correlation_table = pd.DataFrame(correlation_results).sort_values('correlacion', ascending=False).reset_index(drop=True)
    estimated_breed = correlation_table.loc[0, 'raza']
    print('Archivo de prueba:', TEST_AUDIO_PATH)
    print('Raza estimada por correlación:', estimated_breed)
    display(correlation_table)

    test_frequencies, test_spectrum = positive_fft(test_signal)
    figure, axes = plt.subplots(2, 1, figsize=(14, 9))
    axes[0].plot(np.arange(N_SAMPLES) / TARGET_SAMPLE_RATE, test_signal, color='black')
    axes[0].set(title='Señal de prueba filtrada', xlabel='Tiempo [s]', ylabel='Amplitud')
    axes[1].plot(test_frequencies, test_spectrum, color='black', linewidth=2, label='Prueba')
    for breed_name, breed_signal in breed_signals.items():
        frequencies, spectrum = positive_fft(breed_signal)
        axes[1].plot(frequencies, spectrum, alpha=0.75, label=breed_name)
    axes[1].set_xlim(0, 4000)
    axes[1].set(title='Comparación FFT: prueba y patrones de raza', xlabel='Frecuencia [Hz]', ylabel='Magnitud normalizada')
    axes[1].legend()
    axes[1].grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
    display(Audio(test_signal, rate=TARGET_SAMPLE_RATE))

## Conclusiones

1. **Muestreo:** todas las grabaciones se llevan a `8000 Hz` y `8000` muestras para compararlas en las mismas condiciones.
2. **Convolución:** el filtro de media móvil reduce variaciones rápidas y ruido de la señal entrante.
3. **Estadística:** la media, varianza y desviación estándar describen cada señal; la LGN y el TLC justifican el comportamiento de los promedios.
4. **Promedios:** se obtiene primero un patrón para cada perro y después un patrón para cada una de las cinco razas.
5. **Correlación:** la mayor correlación normalizada con la señal de prueba determina la raza estimada. El retardo indica el desplazamiento donde aparece la mayor similitud.
6. **Fourier:** la FFT permite comparar las componentes frecuenciales de la prueba con los cinco patrones.

Este procedimiento no aprende parámetros mediante IA o Machine Learning. Es un método de comparación directa entre señales, construido con las operaciones estudiadas en los laboratorios.